# Ep. 02 — Tokens, embeddings, and a deterministic pipeline

LLMs do not “see” text as characters or words. They see **tokens**. Embeddings project text into numeric vectors so we can compare meaning. Finally we build a small class pipeline that interprets an **ACR** (case/report) and returns structured JSON.


## 1. Tokenization

Limitation: **word** count ≠ **token** count. Spaces and punctuation often stick to the previous piece. Below, a sample string goes through the BPE tokenizer `cl100k_base` (GPT-4 family).


In [ ]:
import tiktoken

text = "Artificial intelligence processes text in tokens, not in words."
enc = tiktoken.get_encoding("cl100k_base")
token_ids = enc.encode(text)
tokens = [enc.decode([tid]) for tid in token_ids]

print("Text:", text)
print("Tokens:", tokens)
print("Token count:", len(tokens))
print("Word count (simple split):", len(text.split()))


## 2. Embeddings (Hugging Face / sentence-transformers)

An embedding is a dense vector that represents the meaning of a text. We use the small model `sentence-transformers/all-MiniLM-L6-v2` (first run downloads the model).


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

word = "joy"
embedding = model.encode(word)

print("Word:", word)
print("Embedding dimension:", len(embedding))
print("First 8 values:", embedding[:8])


## 3. Comparison: opposite meanings

Two texts with opposite meanings land in different regions of vector space. We measure **cosine similarity** (1 ≈ same meaning; low values ≈ far apart).


In [ ]:
import numpy as np

w1, w2 = "joy", "sadness"
e1, e2 = model.encode([w1, w2])

def cosine_similarity(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def euclidean_distance(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.linalg.norm(a - b))

sim = cosine_similarity(e1, e2)
dist = euclidean_distance(e1, e2)

print(f"{w1!r} vs {w2!r}")
print("Cosine similarity:", round(sim, 4))
print("Euclidean distance:", round(dist, 4))


## 4. 3D visualization (PCA)

Embeddings have hundreds of dimensions. For **visualization**, we reduce to 3D with PCA (plot only — real similarity still uses the full vector).


In [ ]:
from sklearn.decomposition import PCA
import plotly.graph_objects as go

labels = ["joy", "sadness", "happiness", "anger", "calm"]
vectors = model.encode(labels)
coords = PCA(n_components=3).fit_transform(vectors)

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=coords[:, 0],
            y=coords[:, 1],
            z=coords[:, 2],
            mode="markers+text",
            text=labels,
            textposition="top center",
            marker=dict(size=6),
        )
    ]
)
fig.update_layout(
    title="Embeddings in 3D (PCA)",
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
    width=700,
    height=500,
)
fig.show()


## 5. Deterministic pipeline: interpret an ACR

Here an **ACR** is a short clinical case/report the student can edit. Two classes run in sequence: `LLMClient` does the OpenAI-style HTTP call, and `ACRInterpreter` requests structured JSON (`category`, `summary`, `urgency`). No LangChain — only `urllib` + `OPENAI_API_KEY`.


In [ ]:
import os
import json
import urllib.request


class LLMClient:
    """Vanilla call to an OpenAI-compatible API."""

    def __init__(self):
        self.api_key = os.environ.get("OPENAI_API_KEY", "").strip()
        self.model = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

    def ask(self, messages):
        if isinstance(messages, str):
            messages = [{"role": "user", "content": messages}]
        if not self.api_key:
            raise SystemExit("Set OPENAI_API_KEY and run again.")
        body = json.dumps(
            {
                "model": self.model,
                "messages": messages,
                "temperature": 0,
                "response_format": {"type": "json_object"},
            }
        ).encode()
        req = urllib.request.Request(
            "https://api.openai.com/v1/chat/completions",
            data=body,
            headers={
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        with urllib.request.urlopen(req) as resp:
            data = json.loads(resp.read().decode())
        return data["choices"][0]["message"]["content"]


class ACRInterpreter:
    """One class calls the other: interpret the ACR via LLMClient."""

    SYSTEM = (
        "You are a clinical assistant. Given an ACR (case report), "
        "reply ONLY with JSON containing keys: "
        "category (string), summary (short string), urgency (low|medium|high)."
    )

    def __init__(self, llm: LLMClient):
        self.llm = llm

    def interpret(self, acr: str) -> dict:
        raw = self.llm.ask(
            [
                {"role": "system", "content": self.SYSTEM},
                {"role": "user", "content": acr},
            ]
        )
        return json.loads(raw)


# Edit the ACR below and run the cell
acr = (
    "67-year-old hypertensive patient arrives with crushing chest pain "
    "for 40 minutes, sweating and dyspnea. BP 90/60 mmHg."
)

result = ACRInterpreter(LLMClient()).interpret(acr)
print(json.dumps(result, ensure_ascii=False, indent=2))


## 6. Docling + OCR (demonstration)

Scanned PDFs often lack a reliable text layer — that is where **OCR** helps. Docling wraps the pipeline: `PdfPipelineOptions(do_ocr=True)` + `DocumentConverter`. The cell below is **demonstration only**: keep `RUN_DOCLING = False` unless you have a local PDF/image and OCR dependencies installed.


In [ ]:
# Demonstration — do NOT run at scale without a local file.
# Set RUN_DOCLING = True and a valid path to try it.

RUN_DOCLING = False
PDF_PATH = "your-scanned-file.pdf"  # PDF or image

if not RUN_DOCLING:
    print("Docling demo off. Set RUN_DOCLING=True and PDF_PATH to extract text.")
else:
    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import PdfPipelineOptions
    from docling.document_converter import DocumentConverter, PdfFormatOption

    pipeline_options = PdfPipelineOptions(do_ocr=True)
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
        }
    )
    result = converter.convert(PDF_PATH)
    text = result.document.export_to_markdown()
    print(text[:2000])


## 7. LM Studio local (Nemotron 3 Nano 4B) — demonstration only

For this demo you must have **LM Studio** running locally with the **Nemotron 3 Nano 4B** model loaded.
The default OpenAI-compatible base URL is `http://localhost:1234/v1`.

This cell is **not part of the default lesson** — demonstration only. Keep `RUN_LM_STUDIO = False` unless LM Studio is running with the model loaded.
Check the exact **model id** in LM Studio (placeholder below: `nvidia/Nemotron-Nano-4B`) and adjust if needed.


In [ ]:
# Demonstration — NOT part of the default lesson.
# Set RUN_LM_STUDIO = True with LM Studio running and Nemotron 3 Nano 4B loaded.

RUN_LM_STUDIO = False
LM_STUDIO_BASE = "http://localhost:1234/v1"
MODEL_ID = "nvidia/Nemotron-Nano-4B"  # adjust to the exact id in LM Studio

if not RUN_LM_STUDIO:
    print("LM Studio demo off. Set RUN_LM_STUDIO=True with the local server running.")
else:
    import json
    import urllib.request

    payload = {
        "model": MODEL_ID,
        "messages": [
            {"role": "user", "content": "Say hello in one short sentence."},
        ],
        "temperature": 0.2,
        "max_tokens": 64,
    }
    req = urllib.request.Request(
        f"{LM_STUDIO_BASE}/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    print(data["choices"][0]["message"]["content"])
